# Employment

Employment by county across stepped years, plus the input city-level employment targets.

In [1]:
import sys
from pathlib import Path

# Make summary_scripts/ importable regardless of where the kernel started.
_HERE = Path.cwd()
for candidate in [_HERE, _HERE / "summary_scripts", _HERE.parent / "summary_scripts"]:
    if (candidate / "validation_data_input.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from validation_data_input import (
    load_config, load_settings, load_unrolled, load_unrolled_regional,
    load_targets, control_id_lookup, load_legacy_pop, load_legacy_units,
    available_years,
)
from util import (
    COUNTY_ID_TO_NAME, COUNTY_ORDER, COUNTY_COLORS, INDICATORS,
    apply_plotly_theme, format_int, pct_diff, style_diff_table, passfail_badge,
)

cfg = load_config()
settings = load_settings()
BASE_YEAR = settings.get("base_year")
END_YEAR = settings.get("end_year")
TARGETS_END_YEAR = settings.get("targets_end_year")

ctrl = load_unrolled().merge(control_id_lookup(), on="control_id", how="left")
ctrl["county"] = ctrl["county_id"].map(COUNTY_ID_TO_NAME)


## Employment by county and year

In [2]:
emp = ctrl.groupby(['county', 'year'])['total_emp'].sum().reset_index()
fig = px.line(
    emp, x='year', y='total_emp', color='county', markers=True,
    category_orders={'county': COUNTY_ORDER},
    color_discrete_map=COUNTY_COLORS,
    labels={'total_emp': 'Jobs', 'year': 'Year', 'county': 'County'},
    title='Total employment by county',
)
apply_plotly_theme(fig)

## Output vs. input city targets (CityEmp)

In [3]:
targets = load_targets('CityEmp')
base_col = f'Emp{BASE_YEAR}'
gro_col = next(c for c in targets.columns if c.startswith('EmpGro'))
ctrl_end = (
    ctrl.loc[ctrl['year'] == TARGETS_END_YEAR, ['control_id', 'total_emp']]
    .rename(columns={'total_emp': 'Emp_ctrl_end'})
)
compare = targets.merge(ctrl_end, on='control_id', how='left')
compare['county'] = compare['county_id'].map(COUNTY_ID_TO_NAME)
by_co = compare.groupby('county').agg(
    target=(gro_col, 'sum'),
    base=(base_col, 'sum'),
    end=('Emp_ctrl_end', 'sum'),
).reindex(COUNTY_ORDER)
by_co['ctrl_growth'] = by_co['end'] - by_co['base']
by_co['delta'] = by_co['ctrl_growth'] - by_co['target']
by_co['% diff vs. target'] = by_co.apply(
    lambda r: pct_diff(r['ctrl_growth'], r['target']), axis=1
)
by_co = by_co.rename(columns={
    'target': 'Target growth',
    'ctrl_growth': 'Controls growth',
    'delta': 'Controls − Target',
})[['Target growth', 'Controls growth', 'Controls − Target', '% diff vs. target']]
style_diff_table(
    by_co,
    int_cols=['Target growth', 'Controls growth', 'Controls − Target'],
    pct_cols=['% diff vs. target'], pct_threshold=2,
)

,Target growth,Controls growth,Controls − Target,% diff vs. target
county,,,,
King,"432,765","432,765",0,+0.0%
Kitsap,"40,681","40,681",0,+0.0%
Pierce,"123,387","123,387",0,+0.0%
Snohomish,"153,377","153,377",0,+0.0%


## Growth (base → targets) by county

In [4]:
growth = ctrl.pivot_table(index='county', columns='year', values='total_emp', aggfunc='sum')
growth = growth.reindex(COUNTY_ORDER)
growth_df = (growth[TARGETS_END_YEAR] - growth[BASE_YEAR]).reset_index()
growth_df.columns = ['county', 'growth']
fig = px.bar(
    growth_df, x='county', y='growth', color='county',
    category_orders={'county': COUNTY_ORDER},
    color_discrete_map=COUNTY_COLORS,
    title=f'Employment growth, {BASE_YEAR}→{TARGETS_END_YEAR}',
    labels={'growth': 'Jobs added', 'county': 'County'},
)
fig.update_layout(showlegend=False)
apply_plotly_theme(fig)

## Notes

- Sector breakdown (e.g. resource/construction split applied in `emp_chg_targets_*.py`) is not in the final controls workbook; to validate sector totals, add a new notebook that reads from the intermediate HDF5 outputs.